# NPE tuning

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import warnings
import sys
sys.path.append('../../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
drop_col = range(16, 32)
theta_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")

theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, drop_col, axis=1)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

torch.Size([1000, 3]) torch.Size([1000, 30])


In [3]:
nan_row = np.where(np.isnan(x_test_numpy))[0]
print(nan_row)

theta_test = theta_test[~np.isnan(x_test_numpy).any(axis=1)]
x_test = x_test[~np.isnan(x_test_numpy).any(axis=1)]
print(theta_test.shape, x_test.shape)

theta_test_numpy = theta_test.cpu().numpy()
x_test_numpy = x_test.cpu().numpy()
print(theta_test_numpy.shape, x_test_numpy.shape)

[865 899]
torch.Size([998, 3]) torch.Size([998, 30])
(998, 3) (998, 30)


In [4]:
theta = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")
x = np.delete(x, drop_col, axis=1)
print(theta.shape, x.shape)

theta = torch.tensor(theta, device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()
print(theta.shape, x.shape)

(10000, 3) (10000, 30)
torch.Size([10000, 3]) torch.Size([10000, 30])


## Test functions

In [5]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    print("Kolmogorov-Smirnov Test Results (SBC):")
    print("-" * 40)

    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)

        print(f"Parameter:        {parameter_labels[dim]}")
        print(f"KS Statistic (D): {ks_stat:.4f}")
        print(f"p-value:          {p_value:.4e}")

        if p_value < 0.05:
            print("Status: MISCALIBRATED (reject null)")
        else:
            print("Status: CALIBRATED (fail to reject null)")
        print("-" * 40)
    
    return ks_results, p_values

In [6]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## Tuning setting

In [7]:
seeds = [1, 2, 3, 4, 5]
num_posterior_samples=1000

## Baseline NPE

In [8]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))
prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [9]:
embedding_net = LeaveLengthOut_NN(
    input_dim=30,
    num_hiddens=48,
    num_hidden_layers=2,
    num_outputs=4)
neural_posterior = posterior_nn(
    model="nsf", 
    embedding_net=embedding_net 
)

In [10]:
seed = seeds[0]
inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
torch.manual_seed(seed)
np.random.seed(seed)

In [11]:
density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
    max_num_epochs=500
)
posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

 Neural network successfully converged after 139 epochs.

In [ ]:
theta_est_post = posterior_baseline.sample_batched((num_posterior_samples,),
                                                   x=x_test,
                                                   show_progress_bars=True,
                                                   reject_outside_prior=False)